# Data Preprocessing for Single-Cell Analysis

This notebook demonstrates the essential preprocessing steps required before running cell segmentation and analysis:

## Key Preprocessing Steps
1. **Dataset Organization**: Standardize file naming and structure
2. **Train-Test Split**: Divide data into training and testing sets for model development
3. **Z-stack Processing**: Combine 2D slices to create 3D volumes for analysis


In [1]:
## Setup and Imports

from pathlib import Path
import json

# Import preprocessing modules
from src.preprocessing import train_test_split_directory
from src.utils.conversion import combine_2d_to_3d
from src.preprocessing.dataset_split import split_dataset_dict

# File handling utilities
from src.utils.file_utils import (
    ImageTimeFileHandler,            # Basic file operations for all files with timestamp
    BF_FileHandler,                # Specialized for brightfield images
    BF_IF_FileHandler               # Specialized for brightfield and IF images
)

## Dataset Organization and Train-Test Split

This section demonstrates how to:

1. **Standardize file naming conventions** - Creates a consistent naming pattern for all images
2. **Split the dataset into training and test sets** - Essential for unbiased model evaluation
3. **Preserve group structure** - Ensures related images (same field of view) stay in the same set

In [2]:
# Define input and output directory paths

# Input data directory - contains original microscopy files 
data_path = "../data/Plate 2426"

# Output directory for 2D preprocessed data
output_dir = "../data/Plate 2426_preprocessed_2D_split_out"

# Output directory for 3D preprocessed data
output_3d_dir = "../data/Plate 2426_preprocessed_3D_split_out"

# Create output directory if it doesn't exist
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(output_3d_dir).mkdir(parents=True, exist_ok=True)

In [6]:
test_split = 0.4
random_seed = 42

# Split the dataset into training and test sets
split_json = train_test_split_directory(
    data_dir = data_path,
    output_dir = output_dir,
    test_size = test_split,
    random_state = random_seed,
    file_handler= ImageTimeFileHandler(),
)

print("Dataset split into train and test sets!")
print(f"Train set: {len(split_json['train_files']['BF'])} images")
print(f"Test set: {len(split_json['test_files']['BF'])} images")


Dataset split into train and test sets!
Train set: 1440 images
Test set: 960 images


## Z-Stack to 3D Volume Conversion

This section demonstrates how to combine 2D z-stack images into 3D volumes for analysis.

The process involves:
1. Identifying related z-slices using filename patterns
2. Sorting slices by z-position
3. Combining slices into a single 3D TIFF file

The pattern parameter uses regular expressions to identify z-stack components in filenames.

In [ ]:
# Combine 2D to 3D
combine_2d_to_3d(
    input_dir=output_dir,
    output_dir=output_3d_dir,
    pattern=r"(.+?)_z(\d+)(?:_(BF|Cells|w2))?\.(tif)",     # Regex pattern to identify z-stacks : will fail without this fixed pattern
    recursive=True,
)


Finding 2D files: 100%|██████████| 7200/7200 [00:00<00:00, 484004.69it/s]


Found 240 groups of 2D images to combine into 3D volumes.
Example groups:
  p2426_I16_t1_w2: 30 files
  p2426_I06_t1_Cells: 30 files
  p2426_C03_t1_Cells: 30 files
  p2426_B08_t1_Cells: 30 files
  p2426_B14_t1_Cells: 30 files


Combining to 3D:  44%|████▍     | 105/240 [01:25<01:36,  1.40it/s]